# vulnRadar — Results Analysis

Report on the matches produced by **vulnRadar**: which tracked repositories later
had a CVE published, how many days after their selection, and how those CVEs are
classified (severity, CVSS, exploitability, CWE).

Data source: `Data/vulnRadar.db` (tables `tracked_repos` and `cve_matches`).

**Sections**
1. Selected repositories (overview per task)
2. Per-repo summary of CVE matches
3. Full match detail (one row per repo/CVE pair)
4. CVE classification
5. Time from selection to CVE — charts

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator

# Path relative to this notebook's folder.
DB = Path('Data') / 'vulnRadar.db'

conn = sqlite3.connect(DB)
df_repos = pd.read_sql_query('SELECT * FROM tracked_repos', conn)
df_matches = pd.read_sql_query('SELECT * FROM cve_matches', conn)
conn.close()

# Matches without a severity are shown as 'n/a' everywhere below.
df_matches['severity'] = df_matches['severity'].fillna('n/a')

print(f'Tracked repos: {len(df_repos):,}')
print(f'CVE matches:   {len(df_matches):,} '
      f'on {df_matches["repo_full_name"].nunique()} distinct repos')

## 1. Selected repositories

How many repositories each selection task contributed, and over which period
they were selected.

In [ ]:
task_overview = (
    df_repos.groupby('task')
    .agg(n_repos=('full_name', 'nunique'),
         first_selection=('selected_date', 'min'),
         last_selection=('selected_date', 'max'))
)
task_overview

## 2. Repositories with at least one CVE match — summary

One row per matched repository:

- `n_cves`: distinct CVEs matched to the repo
- `first_match_days` / `last_match_days`: days between repo selection and CVE publication
- `max_severity`: worst severity among its CVEs
- `mean_cvss`: average CVSS score (ignores CVEs without a score)
- `task`, `score`: which vulnRadar task(s) selected the repo and the selection score

In [ ]:
SEV_ORDER = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']


def max_severity(series):
    ranked = [s for s in series if s in SEV_ORDER]
    return max(ranked, key=SEV_ORDER.index) if ranked else 'n/a'


repo_summary = (
    df_matches.groupby('repo_full_name')
    .agg(n_cves=('cve_id', 'nunique'),
         first_match_days=('days_until_cve', 'min'),
         last_match_days=('days_until_cve', 'max'),
         max_severity=('severity', max_severity),
         mean_cvss=('cvss_score', 'mean'))
    .round({'mean_cvss': 1})
    .sort_values('n_cves', ascending=False)
)

# Add the selecting task(s) and score from tracked_repos.
task_info = (
    df_repos.groupby('full_name')
    .agg(task=('task', lambda s: ', '.join(sorted(set(s)))),
         score=('score', 'max'))
)
repo_summary = repo_summary.join(task_info)
repo_summary

## 3. Full match detail

One row per (repository, CVE) pair, sorted by repo and days-until-CVE.

In [ ]:
detail = (
    df_matches[['repo_full_name', 'cve_id', 'cve_published_date',
                'days_until_cve', 'severity', 'cvss_score',
                'exploitability_score', 'cwe_ids']]
    .sort_values(['repo_full_name', 'days_until_cve', 'cve_id'])
    .reset_index(drop=True)
)
with pd.option_context('display.max_rows', None):
    display(detail)

## 4. CVE classification

Severity distribution, score statistics, and the most frequent CWE categories.
A match can carry several comma-separated CWEs: each one is counted once.

In [ ]:
severity_counts = (
    detail['severity']
    .value_counts()
    .reindex(SEV_ORDER + ['n/a'], fill_value=0)
    .rename('n_matches')
)
print(severity_counts.to_string())
print()
print(detail[['cvss_score', 'exploitability_score']].describe().round(2))

In [ ]:
cwe_counts = (
    detail['cwe_ids']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .rename('n_matches')
)
cwe_counts.head(20)

## 5. Time from selection to CVE

Two views of `days_until_cve` (days between the repo being selected by vulnRadar
and a matching CVE being published):

- **Left — cumulative**: number of repos whose **first** match happened within
  *x* days of selection. Monotonic, reaches the total number of matched repos.
- **Right — first match per day, by task**: on day *x*, how many repos got their
  **first ever** match, split by the task that selected them. Each repo appears
  in exactly one bar (on the day of its first match) — but a repo selected by
  several tasks is counted once **per task**, so the bars can add up to more
  than the number of distinct repos.

In [ ]:
# Fixed colour per task: colour follows the task, never its rank in the chart.
TASK_COLORS = {'official': '#2a78d6', 'hot': '#1baf7a', 'talkers': '#eda100'}

max_day = int(df_matches['days_until_cve'].max())
days = np.arange(0, max_day + 1)

# Day of the FIRST match of each repo.
first_match = df_matches.groupby('repo_full_name')['days_until_cve'].min()

# --- left: cumulative number of repos whose first match happened within x days
cumulative = [(first_match <= d).sum() for d in days]

# --- right: same first-match day, but broken down by selecting task.
# A repo selected by several tasks is counted once per task (by design).
repo_task = df_repos[['full_name', 'task']].drop_duplicates()
firsts_by_task = repo_task.join(first_match, on='full_name', how='inner')

first_by_task = (
    firsts_by_task
    .pivot_table(index='days_until_cve', columns='task',
                 values='full_name', aggfunc='count')
    .reindex(days, fill_value=0)
    .fillna(0)
    .reindex(columns=list(TASK_COLORS), fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].step(days, cumulative, where='post', color='#52514e', linewidth=2)
axes[0].set_title('Repos with first match within x days (cumulative)')
axes[0].set_ylim(0, len(first_match) + 1)

bottom = np.zeros(len(days))
for task, color in TASK_COLORS.items():
    heights = first_by_task[task].to_numpy()
    axes[1].bar(days, heights, bottom=bottom, width=0.8,
                color=color, label=task, edgecolor='white', linewidth=0.5)
    # Direct labels: the palette's lighter hues fall below 3:1 on white, so the
    # count is always readable without relying on colour alone.
    for x, h, b in zip(days, heights, bottom):
        if h > 0:
            axes[1].text(x, b + h / 2, int(h), ha='center', va='center',
                         fontsize=7, color='white', fontweight='bold')
    bottom += heights

axes[1].set_title('Repos matching for the first time on day x, by task')
axes[1].legend(frameon=False, title='Selecting task')
axes[1].set_ylim(0, bottom.max() + 1)

for ax in axes:
    ax.set_xlabel('Days from repo selection to CVE publication')
    ax.set_ylabel('Number of repos')
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlim(-0.5, max_day + 0.5)

plt.tight_layout()
plt.show()

# Table view of the right-hand chart (accessibility: never colour alone).
first_by_task.loc[(first_by_task.sum(axis=1) > 0)].astype(int)